# Design your view

Annotators review faster with an interface fitted to their task: the text large in the middle, the questions
about the whole note next to it, and keyboard shortcuts for the labels they use most. In this tutorial, you turn
the `review_entities` view into such an interface, and see what else `DataWidgetFactory` widgets can show.

!!! info "Before you start"

    Follow [Review entities](review-entities.md) first: this tutorial builds on its `oncology.pret` file and its
    view.

## Add questions about each note

Besides its entities, annotators should tell for each note the site of the cancer, and mark it as reviewed. Add
these two columns to the notes of the file:

In [ ]:
from pret import DataFrame, load_store

store = load_store("views/stores/oncology.pret")
notes = store["notes"].materialize().to_pandas()
store["notes"] = DataFrame(notes.assign(reviewed=False, site=None).astype({"site": "string"}))

`site` starts empty, and its `string` type tells Metanno which values it will hold.

## Write the interface

The new view arranges four panels: the notes on the left, the text in the middle, and on the right a form with the
note questions above the entities of the note:

In [ ]:
%%writefile views/recipes/oncology_review.py
from pret_joy import Divider, Stack
from pret_simple_dock import Layout, Panel

from metanno import DataRelation, DataWidgetFactory

# Name, color and keyboard shortcut of each entity label
LABELS = {
    "tumeur": {"name": "Tumeur", "color": "rgb(255,200,206)", "shortcut": "t"},
    "tnm": {"name": "TNM", "color": "rgb(210,236,247)", "shortcut": "n"},
}
# Choices offered for the cancer site of a note
SITES = ["sein", "poumon", "autre"]


def main(store):
    """
    Review notes with their entities, in panels annotators can rearrange
    """
    factory = DataWidgetFactory(
        store,
        relations=[DataRelation("notes", "entities", on="note_id")],
    )

    # Notes to review, with a checkbox to mark them as reviewed
    notes = factory.create_table_widget(
        store_key="notes",
        primary_key="note_id",
        fields=[
            {"key": "note_id", "kind": "text"},
            {"key": "department", "kind": "text"},
            {"key": "reviewed", "kind": "boolean", "editable": True},
        ],
        style={"height": "100%"},
    )

    # Questions about the selected note, with buttons to move to the
    # previous or next note
    form = factory.create_form_widget(
        store_key="notes",
        primary_key="note_id",
        fields=[
            {"key": "site", "kind": "radio", "editable": True, "options": SITES},
            {"key": "reviewed", "kind": "boolean", "editable": True},
        ],
        add_navigation_buttons=True,
    )

    # Text of the selected note, its label toolbar, and a header showing
    # the ID of the note above the text
    text, toolbar = factory.create_text_widget(
        store_text_key="notes",
        store_spans_key="entities",
        text_key="text",
        text_primary_key="note_id",
        spans_primary_key="entity_id",
        fields=[
            {"key": "label", "kind": "select", "editable": True, "options": list(LABELS)},
            {"key": "negated", "kind": "boolean", "editable": True},
        ],
        labels=LABELS,
    )
    header = factory.create_selected_field_view(
        store_key="notes", shown_key="note_id", fallback="Note"
    )

    # Entities of the selected note
    entities = factory.create_table_widget(
        store_key="entities",
        primary_key="entity_id",
        fields=[
            {"key": "text", "kind": "text"},
            {"key": "label", "kind": "select", "editable": True, "options": list(LABELS)},
            {"key": "negated", "kind": "boolean", "editable": True},
            {"key": "checked", "kind": "boolean", "editable": True},
        ],
        style={"height": "100%"},
    )

    # Panels can be resized, dragged and stacked as tabs. default_config
    # gives their first arrangement, as rows and columns of tab groups
    layout = Layout(
        Panel(notes, key="Notes"),
        Panel(text, key="Text", header=header),
        Panel(Stack(form, Divider(), toolbar, spacing=1), key="Note"),
        Panel(entities, key="Entities"),
        default_config={
            "kind": "row",
            "children": [
                {"tabs": ["Notes"], "size": 20},
                {"tabs": ["Text"], "size": 50},
                {
                    "kind": "column",
                    "size": 30,
                    "children": [
                        {"tabs": ["Note"], "size": 50},
                        {"tabs": ["Entities"], "size": 50},
                    ],
                },
            ],
        },
    )
    # The status bar under the panels reports rejected edits and the
    # connection to Python
    return Stack(layout, factory.create_status_bar(), style={"height": "100%"})

As in `review_entities`, Metanno calls `main` with the Store of the file it opens, the same file the first cell
loaded as `store`, and shows what `main` returns. Compared to that view, three things are new:

- a form, `create_form_widget`, shows the fields of the selected note, with
  <span class="ui-button">← Previous</span> and <span class="ui-button">Next →</span> buttons to go through the notes
- a `shortcut` per label: select some text and press ++t++ to add a tumor, without reaching for the toolbar
- a `Layout` of `Panel`s, from `pret_simple_dock`, the Pret version of
  [react-simple-dock](https://github.com/percevalw/react-simple-dock), instead of a vertical `Stack`

## Open it

In [ ]:
from metanno import view_opener

view_opener(store, view="oncology_review")

The menu of the opener lists both of your views, `oncology_review` and `review_entities`, and opens the one you
choose. Try the new interface:

- drag the title of the **Entities** panel onto **Notes** to stack them as tabs, or resize the panels by dragging
  their borders
- select `poumon` for note 4 in the form, tick `reviewed`, then click <span class="ui-button">Next →</span>
- in the text, select a word and press ++t++, click an entity and press ++backspace++ to delete it, and press
  ++left++ or ++right++ to change notes

Opening the file from the sidebar now uses `oncology_review`, the last view you opened it with.

## Choose how each field is shown

Each entry of `fields` picks a column, and `kind` picks how its values are shown and edited:

| `kind` | Shows | Edited with |
| --- | --- | --- |
| `text` | the value as text | a text box |
| `boolean` | a checkbox | a click |
| `select` | the value | a dropdown of `options` |
| `radio` | the value | radio buttons, in forms |
| `autocomplete` | the value | a searchable list of `options`, for long lists |
| `hyperlink` | a link | not editable |

`editable` allows edits, and `filterable`, `True` by default, shows the filter box of a table column. `options` can
also be a function of the row, to offer different choices for each note:

```python { .no-exec }
def site_options(note):
    """
    Offer the breast site first for radiology notes, mostly mammographies
    """
    return ["sein", "autre"] if note["department"] == "radiologie" else SITES


{"key": "site", "kind": "radio", "editable": True, "options": site_options}
```

??? tip "Working faster in tables"

    - click the ⋮ button of a column header to sort by that column, pin it to the left, or show and hide columns
    - drag a column header onto another one to move it
    - ++ctrl+c++ copies the selected cells, ++ctrl+shift+c++ the selected rows with the column names
    - ++ctrl+z++ undoes the last change made in the view, ++ctrl+shift+z++ redoes it

    On a Mac, use ++cmd++ instead of ++ctrl++.

??? tip "Use your view on database tables"

    The view menu of a source tab in the sidebar also lists your views. To open a view on several database tables
    at once, save a file such as `views/oncology.yml`, which then appears under **Explore** in the sidebar. This
    one assumes your database also holds an `entities` table:

    ```yaml
    recipe: review_entities
    tables:
      notes:
        source: clinic
        table: notes
      entities:
        source: clinic
        table: entities
    ```

    Each entry of `tables` becomes a table of the Store your view receives, read from the database without
    copying it.

## Prepare the next notes for your questions

You added `reviewed` and `site` by hand at the beginning of this tutorial. For the next notes you pick, let
<span class="ui-button">Prepare annotation</span> add them: a view can define `annotation_setup_code`, a function
returning code that the copied cell runs before saving the notes. The source tab must be able to show this view,
so it can only use the `notes` table, like this view of the note questions:

In [ ]:
%%writefile views/recipes/note_questions.py
from pret_simple_dock import Layout, Panel

from metanno import DataWidgetFactory

# Choices offered for the cancer site of a note
SITES = ["sein", "poumon", "autre"]


def main(store):
    """
    Answer questions about each note while reading it
    """
    factory = DataWidgetFactory(store)
    notes = factory.create_table_widget(
        store_key="notes",
        primary_key="note_id",
        fields=[
            {"key": "note_id", "kind": "text"},
            {"key": "department", "kind": "text"},
            {"key": "reviewed", "kind": "boolean", "editable": True},
        ],
        style={"height": "100%"},
    )
    # Without store_spans_key, the text widget only shows the text
    text, _ = factory.create_text_widget(
        store_text_key="notes", text_key="text", text_primary_key="note_id"
    )
    questions = factory.create_form_widget(
        store_key="notes",
        primary_key="note_id",
        fields=[
            {"key": "site", "kind": "radio", "editable": True, "options": SITES},
            {"key": "reviewed", "kind": "boolean", "editable": True},
        ],
        add_navigation_buttons=True,
    )
    return Layout(
        Panel(notes, key="Notes"),
        Panel(text, key="Text"),
        Panel(questions, key="Questions"),
        default_config={
            "kind": "row",
            "children": [
                {"tabs": ["Notes"], "size": 25},
                {"tabs": ["Text"], "size": 50},
                {"tabs": ["Questions"], "size": 25},
            ],
        },
    )


def annotation_setup_code(parameters):
    """
    Add the columns of the note questions to the notes copied by Prepare annotation
    """
    return 'tables["notes"] = tables["notes"].with_columns(reviewed=False, site="")'

Open the `notes` table of the `clinic` source in the sidebar, choose `note_questions` in the view menu of the tab,
then click <span class="ui-button">Copy code</span> → <span class="ui-button">Prepare annotation</span>. The copied
cell now adds the two columns before saving the notes:

```python { .no-exec }
tables = {
    'notes': DataFrame(notes),
}

tables["notes"] = tables["notes"].with_columns(reviewed=False, site="")
```

and the saved file opens with `note_questions`, ready for your annotators.

`annotation_setup_code` receives the `parameters` of the view: the arguments of `main` besides `store`, which
a saved view file gives under `parameters:` and `view_opener` under `parameters=`. They let the setup code
follow the options of the view. `note_questions` has none, so `parameters` is empty and the function ignores it.

## Next

Notes are rarely annotated alone: [Annotate across levels](annotate-across-levels.md) adds the patients above
them, with decisions taken for a whole patient.